# ☁️ Session 10 — Cloud Training Jobs & Experiment Tracking
> **Day 2 | 11:00 AM – 12:00 PM**  
> Submitting jobs to Azure ML • MLflow tracking • Sweep Jobs • Model Registry

## 🎯 What you'll do in this notebook
| # | Skill | Tool |
|---|-------|------|
| 1 | Understand the Job Lifecycle | Matplotlib diagram |
| 2 | Submit a Command Job | Azure ML SDK v2 |
| 3 | Track metrics with MLflow | `mlflow.autolog()` |
| 4 | Create & register an Environment | SDK v2 Entity |
| 5 | Run a Sweep Job (HPO at scale) | SDK v2 Sweep |
| 6 | Visualise sweep results | Parallel coordinates chart |
| 7 | Register the best model | Model Registry |

> ⚠️ **No Azure subscription?** Set `SIMULATION_MODE = True` in Cell 2 — all SDK calls are mocked and all charts render locally.


In [ ]:
# ── Cell 02: Imports & Configuration ─────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

# ── Azure ML SDK v2 (optional) ────────────────────────────────────────────
SIMULATION_MODE = True  # Set False if you have Azure credentials

if not SIMULATION_MODE:
    from azure.ai.ml import MLClient, command, Input, Output
    from azure.ai.ml.entities import AmlCompute, Environment, Model
    from azure.ai.ml.constants import AssetTypes
    from azure.ai.ml.sweep import Choice, Uniform, BanditPolicy
    from azure.identity import DefaultAzureCredential

# ── Workspace config ─────────────────────────────────────────────────────
SUBSCRIPTION_ID = os.getenv('AZURE_SUBSCRIPTION_ID', 'your-subscription-id')
RESOURCE_GROUP  = os.getenv('AZURE_RESOURCE_GROUP',  'ml-resource-group')
WORKSPACE_NAME  = os.getenv('AZURE_ML_WORKSPACE',    'ml-workspace')
COMPUTE_NAME    = 'cpu-cluster'

print('Mode:', 'SIMULATION' if SIMULATION_MODE else 'LIVE AZURE')
print('Workspace:', WORKSPACE_NAME)


## 🔄 The Azure ML Training Job Lifecycle

Every job goes through these stages — understanding this helps you debug failures fast:

1. **Define** — write job spec in Python (SDK v2) or YAML
2. **Upload** — code snapshot pushed to Azure Blob Storage
3. **Provision** — compute cluster auto-scales to add a node
4. **Pull environment** — Docker image pulled from ACR (cached after first run)
5. **Execute** — your `train.py` runs on the node
6. **Log** — MLflow writes metrics/params/artifacts to the tracking store
7. **Save outputs** — model saved back to Blob Storage
8. **Register** — model promoted to Model Registry


In [ ]:
# ── Cell 03 (code): Job Lifecycle Flowchart ──────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.set_xlim(0, 10); ax.set_ylim(0, 3)
ax.axis('off')

stages = [
    (0.6,  1.5, 'Define\nJob',      '#0078D4'),
    (1.9,  1.5, 'Upload\nCode',     '#00BCF2'),
    (3.1,  1.5, 'Provision\nNode',  '#008575'),
    (4.4,  1.5, 'Pull\nEnv (ACR)',  '#107C41'),
    (5.7,  1.5, 'Execute\nScript',  '#744DA9'),
    (7.0,  1.5, 'Log\nMLflow',      '#C239B3'),
    (8.3,  1.5, 'Save\nOutputs',    '#E74856'),
    (9.4,  1.5, 'Register\nModel',  '#FF8C00'),
]

for x, y, label, color in stages:
    ax.add_patch(plt.Circle((x, y), 0.42, color=color, zorder=3))
    ax.text(x, y, label, ha='center', va='center', fontsize=7.5,
            fontweight='bold', color='white', zorder=4)

for i in range(len(stages)-1):
    x1, y1 = stages[i][0] + 0.43,  stages[i][1]
    x2, y2 = stages[i+1][0] - 0.43, stages[i+1][1]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

ax.set_title('Azure ML Training Job Lifecycle', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()
print('Stages: Define → Upload → Provision → Pull Env → Execute → Log → Save → Register')


## 🚀 Submitting a Command Job (SDK v2)

A **Command Job** is the simplest job type — it runs a single script on a compute cluster.

```python
job = command(
    code='./scripts',
    command='python train.py --learning-rate 0.05 --max-depth 5',
    environment='azureml:credit-model-env:1',
    compute='cpu-cluster',
    experiment_name='credit-default-model'
)
returned_job = ml_client.jobs.create_or_update(job)
print('Job URL:', returned_job.studio_url)
```

**Navigate in Studio:** `Jobs` → select experiment → click your run → view logs & metrics.


In [ ]:
# ── Cell 04 (code): Submit Command Job ───────────────────────────────────
if not SIMULATION_MODE:
    credential = DefaultAzureCredential()
    ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

    job = command(
        code='./scripts',
        command=(
            'python train.py '
            '--learning-rate 0.05 '
            '--max-depth 5 '
            '--n-estimators 200'
        ),
        environment='azureml:credit-model-env:1',
        compute=COMPUTE_NAME,
        display_name='credit-model-training-v1',
        experiment_name='credit-default-model',
        tags={'framework': 'sklearn', 'session': '10'}
    )
    returned_job = ml_client.jobs.create_or_update(job)
    print('Submitted job:', returned_job.name)
    print('Studio URL:',    returned_job.studio_url)
else:
    import uuid
    fake_job_name = f'credit_training_{str(uuid.uuid4())[:8]}'
    print('[SIMULATION] Job submitted (mocked)')
    print(f'[SIMULATION] Job name: {fake_job_name}')
    print(f'[SIMULATION] Studio URL: https://ml.azure.com/runs/{fake_job_name}')


## 📊 Experiment Tracking with MLflow

Azure ML automatically sets the MLflow tracking URI — your `train.py` doesn't need any config.

### Auto-logging vs Manual

| Feature | `mlflow.autolog()` | Manual `mlflow.log_*` |
|---------|-------------------|----------------------|
| Setup effort | One line | Explicit per metric |
| Params captured | All sklearn params | You choose |
| Metrics captured | train/val score | You define + steps |
| Custom artifacts | ❌ | ✅ (figures, JSON) |
| Best for | Quick experiments | Production pipelines |

### What mlflow.autolog() captures for sklearn:
- All `__init__` parameters (`n_estimators`, `max_depth`, etc.)
- Training score
- Feature importances (as artifact)
- The fitted model itself


In [ ]:
# ── Cell 06 (code): Training Script with MLflow Logging (Local Simulation) ─
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, confusion_matrix

# Generate synthetic credit data
np.random.seed(42)
n = 4000
X_raw = pd.DataFrame({
    'limit_bal':  np.random.randint(10000, 500000, n),
    'age':        np.random.randint(21, 75, n),
    'pay_0':      np.random.randint(-1, 9, n),
    'pay_2':      np.random.randint(-1, 9, n),
    'bill_amt1':  np.random.randint(0, 200000, n),
    'pay_amt1':   np.random.randint(0, 50000, n),
})
y_raw = ((0.3*(X_raw['pay_0']>2) + 0.2*(X_raw['limit_bal']<50000) +
          np.random.rand(n)*0.5) > 0.55).astype(int)

X_train, X_val, y_train, y_val = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Local MLflow run
mlflow.set_experiment('credit-default-model-local')

with mlflow.start_run(run_name='cmd-job-sim') as run:
    params = {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'min_samples_split': 20}
    mlflow.log_params(params)

    model = GradientBoostingClassifier(**params, random_state=42)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('val_auc',   val_auc)

    # Confusion matrix
    cm = confusion_matrix(y_val, model.predict(X_val))
    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=14, fontweight='bold')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Pred 0','Pred 1']); ax.set_yticklabels(['Act 0','Act 1'])
    ax.set_title('Confusion Matrix', fontweight='bold')
    fig.colorbar(im); plt.tight_layout()
    mlflow.log_figure(fig, 'confusion_matrix.png')
    plt.show(); plt.close()

    run_id = run.info.run_id

print(f'Train AUC: {train_auc:.4f}')
print(f'Val AUC:   {val_auc:.4f}')
print(f'MLflow Run ID: {run_id}')


## 🐋 Azure ML Environments

An **Environment** bundles your Python packages + Docker base image into a versioned, reproducible unit. Azure ML caches the built image in ACR so subsequent jobs start faster.

### Why Environments Matter
- **Reproducibility** — same image = same result, 6 months later
- **Speed** — first build ~10 min, subsequent runs pull from ACR cache (<1 min)
- **Auditability** — every model version links to the exact environment it was trained with

```yaml
# environment.yml
name: credit-model-env
dependencies:
  - python=3.10
  - pip:
    - scikit-learn==1.3.0
    - pandas==2.0.3
    - mlflow==2.7.0
```

**Curated environments** — Azure ML ships pre-built images you can start from:
- `azureml:sklearn-1.0-ubuntu20.04-py38-cpu:1`
- `azureml:pytorch-2.0-gpu-py38:1`
- Browse at: **Studio → Environments → Curated**


In [ ]:
# ── Cell 08 (code): Create & Register Custom Environment ─────────────────
SCRIPTS_DIR = os.path.join(os.path.dirname(os.getcwd()), '02_cloud_training_experiment', 'scripts')
CONDA_FILE = os.path.join(SCRIPTS_DIR, 'environment.yml')

if not SIMULATION_MODE:
    env = Environment(
        name='credit-model-env',
        version='1',
        description='Environment for credit default model - Session 10',
        conda_file=CONDA_FILE,
        image='mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest'
    )
    registered_env = ml_client.environments.create_or_update(env)
    print(f'Registered environment: {registered_env.name} v{registered_env.version}')
else:
    print('[SIMULATION] Environment registration mocked')
    print('  Name:    credit-model-env')
    print('  Version: 1')
    print('  Base:    mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest')
    if os.path.exists(CONDA_FILE):
        print(f'  Conda file found at: {CONDA_FILE}')
        with open(CONDA_FILE) as f:
            print('--- environment.yml ---')
            print(f.read())


## 🔍 Sweep Jobs — Cloud-Scale Hyperparameter Optimization

A **Sweep Job** converts any Command Job into a parallelized HPO run on Azure ML.

### Key Components
| Component | What it does |
|-----------|-------------|
| `sampling_algorithm` | How to pick hyperparams: `random`, `grid`, `bayesian` |
| `search_space` | Parameter distributions: `Choice`, `Uniform`, `LogUniform` |
| `primary_metric` | The metric to optimize (must be logged in your script) |
| `goal` | `Maximize` or `Minimize` |
| `max_total_trials` | Hard cap on total runs (cost control) |
| `max_concurrent_trials` | How many nodes to use simultaneously |
| `early_termination_policy` | **Bandit Policy** — kills trials >10% behind the best |

### Why Bayesian > Random > Grid
- **Grid** — exhaustive; expensive for large spaces
- **Random** — no learning between trials
- **Bayesian** — learns from previous results; converges ~5x faster than random


In [ ]:
# ── Cell 10 (code): Define & Submit Sweep Job ────────────────────────────
if not SIMULATION_MODE:
    from azure.ai.ml.sweep import Choice, Uniform, BanditPolicy

    command_job = command(
        code='./scripts',
        command=(
            'python train.py '
            '--learning-rate ${{search_space.learning_rate}} '
            '--max-depth ${{search_space.max_depth}} '
            '--n-estimators ${{search_space.n_estimators}}'
        ),
        environment='azureml:credit-model-env:1',
        compute=COMPUTE_NAME,
        experiment_name='credit-default-sweep',
    )

    sweep_job = command_job.sweep(
        sampling_algorithm='bayesian',
        primary_metric='val_auc',
        goal='Maximize',
        search_space={
            'learning_rate': Uniform(min_value=0.001, max_value=0.1),
            'max_depth':     Choice(values=[3, 5, 7, 10]),
            'n_estimators':  Choice(values=[100, 200, 300, 500]),
        },
        limits={'max_total_trials': 30, 'max_concurrent_trials': 5, 'timeout': 7200},
        early_termination_policy=BanditPolicy(slack_factor=0.1, evaluation_interval=1)
    )
    returned_sweep = ml_client.jobs.create_or_update(sweep_job)
    print('Sweep Job:', returned_sweep.name)
    print('Studio URL:', returned_sweep.studio_url)
else:
    # Simulate 30 trials with random search spaces
    np.random.seed(99)
    sim_trials = []
    for i in range(30):
        lr  = np.random.uniform(0.001, 0.1)
        md  = np.random.choice([3, 5, 7, 10])
        ne  = np.random.choice([100, 200, 300, 500])
        m = GradientBoostingClassifier(
            learning_rate=lr, max_depth=md, n_estimators=ne, random_state=i
        )
        m.fit(X_train, y_train)
        auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        sim_trials.append({'trial': i+1, 'learning_rate': lr,
                           'max_depth': md, 'n_estimators': ne, 'val_auc': auc})
    sweep_df = pd.DataFrame(sim_trials)
    print(f'[SIMULATION] {len(sweep_df)} trials completed')
    print(sweep_df.sort_values('val_auc', ascending=False).head(5).to_string(index=False))


In [ ]:
# ── Cell 11 (code): Parallel Coordinates Chart — Sweep Results ──────────
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

if 'sweep_df' not in dir():
    # Fallback if cell 10 was skipped
    np.random.seed(99)
    sim_trials = []
    for i in range(30):
        lr = np.random.uniform(0.001, 0.1)
        md = np.random.choice([3, 5, 7, 10])
        ne = np.random.choice([100, 200, 300, 500])
        m = GradientBoostingClassifier(
            learning_rate=lr, max_depth=md, n_estimators=ne, random_state=i)
        m.fit(X_train, y_train)
        auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        sim_trials.append({'trial':i+1,'learning_rate':lr,'max_depth':md,
                           'n_estimators':ne,'val_auc':auc})
    sweep_df = pd.DataFrame(sim_trials)

axes_labels = ['learning_rate', 'max_depth', 'n_estimators', 'val_auc']
fig, axes = plt.subplots(1, len(axes_labels)-1, figsize=(12, 5), sharey=False)
fig.suptitle('Sweep Job — Parallel Coordinates (30 trials)', fontsize=14, fontweight='bold')

norm = Normalize(vmin=sweep_df['val_auc'].min(), vmax=sweep_df['val_auc'].max())
cmap = plt.cm.RdYlGn

for _, row in sweep_df.iterrows():
    color = cmap(norm(row['val_auc']))
    vals = [row[col] for col in axes_labels]
    for i, ax in enumerate(axes):
        # Normalize each axis to [0,1] for plotting
        col1, col2 = axes_labels[i], axes_labels[i+1]
        mn1, mx1 = sweep_df[col1].min(), sweep_df[col1].max()
        mn2, mx2 = sweep_df[col2].min(), sweep_df[col2].max()
        y1 = (row[col1] - mn1) / (mx1 - mn1 + 1e-9)
        y2 = (row[col2] - mn2) / (mx2 - mn2 + 1e-9)
        ax.plot([0, 1], [y1, y2], color=color, alpha=0.5, lw=1.2)

for i, ax in enumerate(axes):
    ax.set_xticks([0, 1])
    ax.set_xticklabels([axes_labels[i], axes_labels[i+1]], fontsize=9)
    ax.set_ylim(0, 1); ax.set_yticks([])
    ax.spines[['top','bottom']].set_visible(False)

sm = ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=axes, shrink=0.7, pad=0.02)
cbar.set_label('val_auc', fontsize=10)

plt.tight_layout()
plt.show()

best = sweep_df.loc[sweep_df['val_auc'].idxmax()]
print(f'Best trial: lr={best.learning_rate:.4f}, max_depth={best.max_depth}, '
      f'n_estimators={int(best.n_estimators)}, val_auc={best.val_auc:.4f}')


## 📦 Model Registry — Single Source of Truth

The **Azure ML Model Registry** gives every registered model:
- A **unique name + version** (auto-incremented)
- **Tags** — link it to metrics, datasets, frameworks
- **Labels** — `None → Staging → Production` promotion
- **Lineage** — traces back to the exact run, environment, and code that produced it

### Navigate in Studio
1. **Azure ML Studio → Models**
2. Select a model → click **Version history**
3. Click a version → see lineage, tags, and artifacts
4. Click **Deploy** → create a managed online/batch endpoint

### Promotion Workflow
```
Experiment run ──→ Register (None) ──→ Staging (QA testing) ──→ Production
                                               ↑                      ↑
                              CI/CD pipeline validates          Champion swap
```


In [ ]:
# ── Cell 13 (code): Register Best Model + State Transition Diagram ───────
best = sweep_df.loc[sweep_df['val_auc'].idxmax()]

if not SIMULATION_MODE:
    from azure.ai.ml.entities import Model
    from azure.ai.ml.constants import AssetTypes

    model_entity = Model(
        path=f'azureml://jobs/{returned_sweep.name}/outputs/artifacts/named-outputs/model',
        name='credit-default-classifier',
        description='GBM credit default model — best sweep trial',
        type=AssetTypes.MLFLOW_MODEL,
        tags={
            'val_auc':    str(round(float(best.val_auc), 4)),
            'framework':  'sklearn',
            'session':    '10',
        }
    )
    reg = ml_client.models.create_or_update(model_entity)
    print(f'Registered: {reg.name} v{reg.version}')
else:
    print('[SIMULATION] Model registration mocked')
    print(f'  Name:    credit-default-classifier')
    print(f'  Version: 1')
    print(f'  val_auc: {best.val_auc:.4f}')
    print(f'  Tags:    framework=sklearn, session=10')

# State Transition Diagram
fig, ax = plt.subplots(figsize=(11, 3))
ax.axis('off')
ax.set_xlim(0, 11); ax.set_ylim(0, 3)

states = [
    (1.2, 1.5, 'Experiment\nRun', '#6B7280'),
    (3.5, 1.5, 'Registered\n(None)', '#0078D4'),
    (6.0, 1.5, 'Staging', '#107C41'),
    (8.5, 1.5, 'Production', '#FF8C00'),
    (10.3,1.5, 'Archived', '#C4C4C4'),
]
for x, y, label, color in states:
    ax.add_patch(plt.Circle((x, y), 0.55, color=color, zorder=3))
    ax.text(x, y, label, ha='center', va='center', fontsize=8,
            fontweight='bold', color='white', zorder=4)

transitions = [
    (states[0], states[1], 'Register'),
    (states[1], states[2], 'Set label:\nStaging'),
    (states[2], states[3], 'Set label:\nProduction'),
    (states[3], states[4], 'Deprecate'),
]
for (x1,y1,_,__), (x2,y2,_,__), label in transitions:
    ax.annotate('', xy=(x2-0.56, y2), xytext=(x1+0.56, y1),
                arrowprops=dict(arrowstyle='->', color='#444', lw=1.5))
    mx = (x1+x2)/2
    ax.text(mx, 2.15, label, ha='center', fontsize=7.5, color='#333')

ax.set_title('Azure ML Model Registry — State Transitions', fontsize=13, fontweight='bold', pad=8)
plt.tight_layout()
plt.show()


## ✅ Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **Command Job** | Wrap any script in a versioned, compute-tracked cloud run |
| **`mlflow.autolog()`** | One line → full param/metric/model tracking for sklearn |
| **Sweep Job** | 30 parallel trials with Bayesian sampling + Bandit termination = fast, cheap HPO |
| **Environments** | Docker + conda, versioned and cached in ACR |
| **Model Registry** | Every model has a version, tags, lineage, and promotion path |
| **Studio Navigation** | Jobs → Experiments → Models → Compute → Environments |

---

## ❓ Quiz

1. What is the difference between a **Command Job** and a **Sweep Job** in Azure ML?
2. What does `mlflow.autolog()` capture automatically for sklearn models?
3. Why is the **Bandit early termination policy** important for cost control in Sweep Jobs?
4. What does it mean to "register" a model — what information is stored?
5. If `max_concurrent_trials=5` and `max_total_trials=30`, how many 'rounds' will the sweep take?
6. Where in Azure ML Studio do you compare multiple training runs side-by-side?
7. Why do Environments get cached in ACR after the first build?

---

## 📖 References
- [Azure ML: Submit a Training Job](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-train-model)
- [Hyperparameter Tuning with Sweep Jobs](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-tune-hyperparameters)
- [MLflow on Azure ML](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-use-mlflow-cli-runs)
- [Azure ML Model Registry](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-models)
